In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_date AS
WITH all_dates AS (
    SELECT date_received        AS d FROM silver.customer_complaints WHERE date_received IS NOT NULL
    UNION
    SELECT date_sent_to_company AS d FROM silver.customer_complaints WHERE date_sent_to_company IS NOT NULL
)
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
    d                                       AS full_date,
    year(d)                                 AS year,
    quarter(d)                              AS quarter,
    month(d)                                AS month,
    date_format(d, 'MMMM')                  AS month_name
FROM all_dates;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_product AS
SELECT DISTINCT
    xxhash64(product, sub_product) AS product_key,
    product,
    sub_product
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_issue AS
SELECT DISTINCT
    xxhash64(issue, sub_issue) AS issue_key,
    issue,
    sub_issue
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_company AS
SELECT DISTINCT
    xxhash64(company) AS company_key,
    company
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_geography AS
SELECT DISTINCT
    xxhash64(state, zip3) AS geography_key,
    state,
    zip3
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_submission AS
SELECT DISTINCT
    xxhash64(submitted_via, consumer_consent_provided, tags) AS submission_key,
    submitted_via,
    consumer_consent_provided,
    tags
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_response AS
SELECT DISTINCT
    xxhash64(company_response_to_consumer, company_public_response) AS response_key,
    company_response_to_consumer,
    company_public_response
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.fact_complaints AS
SELECT
    -- degenerate dimension (natural key)
    complaint_id,

    -- foreign keys
    CAST(date_format(date_received,        'yyyyMMdd') AS INT)        AS date_received_key,
    CAST(date_format(date_sent_to_company, 'yyyyMMdd') AS INT)        AS date_sent_key,
    xxhash64(product, sub_product)                                    AS product_key,
    xxhash64(issue, sub_issue)                                        AS issue_key,
    xxhash64(company)                                                 AS company_key,
    xxhash64(state, zip3)                                             AS geography_key,
    xxhash64(submitted_via, consumer_consent_provided, tags)          AS submission_key,
    xxhash64(company_response_to_consumer, company_public_response)   AS response_key,

    -- measures
    days_to_send,
    is_timely_response,
    is_in_progress,
    has_narrative,
    narrative_length,

    -- audit / lineage
    ingestion_timestamp,
    source_system
FROM silver.customer_complaints;